# Benchmark 3: 18 mJ, 1.3 ps, argon-filled Herriott cell

**Paper:** M. Kaumanns, V. Pervak, D. Kormin, V. Leshchenko, A. Kessel, M. Ueffing, Y. Chen, T. Nubbemeyer, *Multipass spectral broadening of 18 mJ pulses compressible from 1.3 ps to 41 fs*, Opt. Lett. **43**, 5877 (2018), [doi:10.1364/OL.43.005877](https://doi.org/10.1364/OL.43.005877).

**Why this case:** the high-energy regime: 13 GW peak power (P/Pc ≈ 0.5), a large total nonlinear phase (a fit of the measured spectrum gives ≈ 55 rad), a compression factor of 33, and a near-concentric cell close to its stability edge. The authors also give a simple SPM model of their output (Eq. 1 with φ = 55 rad), which is plotted here as a second reference.

| | paper | used here |
|---|---|---|
| mirrors | rectangular 300 × 130 mm, RoC 1.5 m, R > 99.9 % over 965–1100 nm, GDD = 0 at 1030 nm, \|GDD\| < 50 fs² | R = 0.999, GDD = 0, edges 10 nm outside the band (assumption) |
| geometry | d ≈ 1.99 R ≈ 3 m, 45 passes, elliptical spot pattern | `MPC(R=1.5, d=2.986, n_passes=45)`; d chosen to give the quoted eigenmode |
| eigenmode | 0.37 mm diameter at focus, 5.4 mm on mirror | computed below |
| gas | Ar 600 mbar, n₂(1 bar) ≈ (10 ± 1)×10⁻²⁴ m²/W | same |
| input | 18.6 mJ, 1.3 ps FWHM, 1.1 nm bandwidth, 1030 nm | Gaussian, transform-limited 1.3 ps |
| compressor | 17 chirped mirrors, total −9400 fs² | same GDD |

**Measured results to compare with:** 17.8 mJ output (95.7 %), Fourier limit 39 fs, compressed 41 fs with 62 % of the Fourier-limited peak power (their model: 41.6 fs, 67 %), M² 1.17 × 1.18, intensity-weighted homogeneity 96.7 % (x) and 97.3 % (y). A variant at 700 mbar and 17 mJ produced new components near 950 nm (set `pressure_bar=0.7`, `energy=17e-3`, `lambda_min=920e-9` to try it).

**Not in the model:** ionisation (the experiment runs just below the measured breakdown energy, ~18.3 mJ at 600 mbar), and the ASE / pre- and post-pulse background that the authors identify as the cause of the central peak of the measured spectrum.

In [ ]:
import os, time, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
import thunder4d as th
from thunder4d import diagnostics as dg

FS = 1e-15
plt.rcParams.update({"figure.dpi": 110})
print("thunder4d", th.__version__, "| GPU available:", th.gpu_available())

In [ ]:
# ---------------- run settings ----------------
QUICK = False          # True: 2 passes on a coarse grid, only to check that the notebook runs
DEVICE = "auto"        # "auto" (GPU if CuPy works), "gpu" or "cpu"
CPU_BUDGET_GB = 16     # memory the run may use when it runs on the CPU
PHI_MAX = 0.02         # peak nonlinear phase per step: halve it once to check convergence
MAX_SECONDS = None     # e.g. 3600: stop cleanly after ~1 h, re-run the run cell to continue
RESUME = True          # continue from the checkpoint of an identical run if there is one

In [ ]:
# ---------------- helpers (same in the three benchmark notebooks) ----------------
def device_budget_gb():
    """Memory the run may use: 80 % of the free GPU memory, or CPU_BUDGET_GB on the CPU."""
    if DEVICE != "cpu" and th.gpu_available():
        import cupy as cp
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
        return 0.8 * free / 1e9
    print(f"CPU run, budget {CPU_BUDGET_GB} GB")
    return CPU_BUDGET_GB


def pick_grid(cell, presets, budget_gb, n_fields=12, **grid_args):
    """Finest preset whose estimated memory (about n_fields x one field) fits in budget_gb."""
    for p in presets:
        g = th.suggest_grid(cell, **grid_args, **p, device="cpu")       # axes only: cheap
        need = n_fields * np.prod(g.shape) * 8 / 1e9                    # complex64
        ok = need <= budget_gb
        print(f"  {p}: {g.Nx}x{g.Ny}x{g.Nt}, ~{need:.1f} GB" + ("  <- used" if ok else ""))
        if ok:
            return th.suggest_grid(cell, **grid_args, **p, device=DEVICE), p
    raise MemoryError("no preset fits the memory budget: add a coarser preset or use a bigger GPU")


def band_mirror(R_in, lam_lo, lam_hi, R_out=0.0, order=80):
    """Intensity reflectivity R(lambda) [lambda in m]: R_in inside the band, smooth edges with their
    50 % points at lam_lo and lam_hi, R_out far outside."""
    lc, hw = 0.5 * (lam_lo + lam_hi), 0.5 * (lam_hi - lam_lo)
    def R(lam):
        u = np.abs((np.asarray(lam, float) - lc) / hw)
        return R_out + (R_in - R_out) * np.exp(-np.log(2) * u ** order)
    return R


def gdd_phase(grid, gdd_fs2=0.0, tod_fs3=0.0):
    """Spectral phase on grid.Omega (FFT order) of a GDD [fs^2] and TOD [fs^3]."""
    W = grid.Omega * FS
    return gdd_fs2 / 2 * W ** 2 + tod_fs3 / 6 * W ** 3


def beam_pixels(Aw, rel=1e-4):
    """(npix, Nt) host array of the pixels that carry the beam (fluence > rel x max)."""
    A = th.to_host(Aw)
    F = np.sum(np.abs(A) ** 2, axis=2)
    return A[F > rel * F.max()]


def _pad(E, up):
    """Zero-pad spectra (FFT order, last axis) by `up` -> time step dt/up."""
    if up == 1:
        return E
    N = E.shape[-1]
    h = (N + 1) // 2
    out = np.zeros(E.shape[:-1] + (N * up,), complex)
    out[..., :h] = E[..., :h]
    out[..., N * up - (N - h):] = E[..., h:]
    return out


def integrated_power(grid, E_pix, phase=None, up=4):
    """Spatially integrated P(t) [W] of E_pix (npix, Nt) after the spectral phase `phase`
    (array on grid.Omega). Returns sorted t [fs] and P [W], time step dt/up."""
    E = E_pix if phase is None else E_pix * np.exp(1j * phase)[None, :]
    A = grid.hfft.ift_t(_pad(E, up)) * up
    P = np.sum(np.abs(A) ** 2, axis=0) * grid.dA
    t = np.fft.fftfreq(grid.Nt * up, 1.0 / (grid.Nt * grid.dt))
    return np.fft.fftshift(t) / FS, np.fft.fftshift(P)


def fourier_limit(grid, E_pix, up=4):
    """Fourier limit of the spatially integrated spectrum (the 'Fourier limit' quoted in papers)."""
    S = np.sum(np.abs(E_pix) ** 2, axis=0)
    return integrated_power(grid, np.sqrt(S)[None, :], None, up)


def fwhm(x, y):
    """FWHM of the main peak (linear interpolation)."""
    y = np.asarray(y, float)
    i = int(np.argmax(y)); h = y[i] / 2
    l, r = i, i
    while l > 0 and y[l] > h: l -= 1
    while r < len(y) - 1 and y[r] > h: r += 1
    xl = np.interp(h, [y[l], y[l + 1]], [x[l], x[l + 1]])
    xr = np.interp(h, [y[r], y[r - 1]], [x[r], x[r - 1]])
    return xr - xl


def span(lam, S, level_db=-20):
    """(lambda_min, lambda_max) [nm] of the outermost points above level_db relative to the peak."""
    ok = np.nonzero(S / S.max() >= 10 ** (level_db / 10))[0]
    return lam[ok[0]] * 1e9, lam[ok[-1]] * 1e9


def homogeneity_1d(S, rel=np.exp(-2)):
    """Spectral homogeneity as defined by Weitenberg et al. (Opt. Express 25, 20502, 2017) and used in
    the papers: V(x) = (int sqrt(S S_ref))^2 / (int S int S_ref), S_ref = spectrum at the intensity
    maximum. S[x, lambda]. Returns V(x) and its intensity-weighted mean inside the 1/e^2 width."""
    prof = S.sum(axis=1)
    S0 = S[np.argmax(prof)]
    V = np.sum(np.sqrt(S * S0[None, :]), axis=1) ** 2 / np.maximum(S.sum(axis=1) * S0.sum(), 1e-300)
    inside = prof > rel * prof.max()
    return V, float(np.sum(V[inside] * prof[inside]) / np.sum(prof[inside]))


def load_digitized(path):
    """Optional measured curve digitised from a paper figure: CSV with 2 columns (x, y)."""
    if not os.path.exists(path):
        print(f"(no {path}: digitise the figure, e.g. with WebPlotDigitizer, to overlay the measurement)")
        return None
    d = np.loadtxt(path, delimiter=",")
    d = d[np.argsort(d[:, 0])]
    return d[:, 0], d[:, 1] / d[:, 1].max()


def table(rows):
    """rows: (quantity, simulation, paper) -> aligned text table."""
    print(f"{'quantity':42s}{'simulation':>16s}{'paper':>22s}")
    print("-" * 80)
    for q, s, p in rows:
        print(f"{q:42s}{s:>16s}{p:>22s}")


def run_tag(params, grid):
    """Short fingerprint of the parameters and grid, used to name the checkpoint."""
    s = json.dumps({k: str(v) for k, v in sorted(params.items())}) + str(grid.shape) + str(PHI_MAX) + str(QUICK)
    return hashlib.md5(s.encode()).hexdigest()[:8]

## 1. Parameters from the paper

In [ ]:
P = dict(
    lambda0=1030e-9,
    energy=18.6e-3,          # paper
    fwhm=1.3e-12,            # paper (Fourier-limited input)
    pressure_bar=0.6,        # paper
    n2_1bar=1.0e-23,         # paper: (10 +- 1) x 1e-24 m^2/W at 1 bar
    R=1.5, d=2.986,          # paper: RoC 1.5 m, d ~ 1.99 R; 2.986 m reproduces the quoted eigenmode
    n_passes=45,             # paper
    mirror_R=0.999,          # paper: R > 99.9 % over 965-1100 nm
    band=(955e-9, 1110e-9),  # assumption: 50 % edges 10 nm outside the specified band
    compressor_gdd=-9400.0,  # paper
    phi_fit=55.0,            # paper: phase of the SPM model fitted to the measured spectrum
    lambda_min=960e-9, lambda_max=1110e-9,
    injection="linear",
)
REF = dict(w0_um=185, wm_um=2700, T=0.957, tl_fs=39, comp_fs=41, peak=0.62, M2="1.17 / 1.18",
           V="96.7 / 97.3 %")
RECORD_EVERY = 5             # full diagnostics every 5 passes (they are costly on this large grid)
RUN_DIR = "runs_bench3"
os.makedirs(RUN_DIR, exist_ok=True)

## 2. Gas, cell and eigenmode

In [ ]:
gas = th.Material("Ar", P["pressure_bar"], n2_1bar=P["n2_1bar"])
cell = th.MPC(R=P["R"], d=P["d"], medium=gas, n_passes=2 if QUICK else P["n_passes"],
              mirror_reflectivity=band_mirror(P["mirror_R"], *P["band"]))
lin = cell.linear_mode(P["lambda0"])
print(cell.report())
print(f"linear eigenmode: waist {lin['w0']*1e6:.0f} um (paper {REF['w0_um']}), "
      f"mirror {lin['w_mirror']*1e6:.0f} um (paper {REF['wm_um']})  [1/e^2 radii]")
print(f"Pc = {gas.critical_power(P['lambda0'])/1e9:.1f} GW, GDD of 45 passes in the gas: "
      f"{gas.gvd(P['lambda0']) * P['d'] * P['n_passes']:.0f} fs^2 (paper: ~1150 fs^2)")

## 3. Grid
This is the largest case: a 2.7 mm mirror spot next to a 185 µm waist, and a picosecond pulse broadened 30 times. The presets trade resolution for memory; the finest that fits is used.

In [ ]:
PRESETS = ([dict(box_factor=4, points_per_waist=3, time_window_factor=3.5)] if QUICK else
           [dict(box_factor=7, points_per_waist=7, time_window_factor=6),
            dict(box_factor=6, points_per_waist=6, time_window_factor=5),
            dict(box_factor=5, points_per_waist=5, time_window_factor=4.5),
            dict(box_factor=4.5, points_per_waist=4.5, time_window_factor=4)])
grid, preset = pick_grid(cell, PRESETS, device_budget_gb(), lambda0=P["lambda0"],
                         lambda_min=P["lambda_min"], lambda_max=P["lambda_max"], pulse_fwhm=P["fwhm"])
print(grid.describe())

## 4. Input pulse and injection

In [ ]:
E = th.gaussian_spectrum(grid, fwhm_fs=P["fwhm"] / FS)
probe = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(lin["w0"])), P["energy"])
mm = cell.match(probe, verbose=True)
w0_ref = lin["w0"] if P["injection"] == "linear" else mm["w0"]
pulse = th.Pulse.at_focus(grid, E, th.Beam(th.gaussian(w0_ref), chromatic_size="eigenmode"), P["energy"])
t, Pt = dg.temporal_power(grid, pulse.Aw)
print(f"injected waist {w0_ref*1e6:.0f} um ({P['injection']}), peak power {Pt.max()/1e9:.1f} GW, "
      f"FWHM {fwhm(t / FS, Pt):.0f} fs")

## 5. Run

In [ ]:
sim = th.MPCSimulation(pulse, cell, phi_max=PHI_MAX, mode_reference=w0_ref, near_field=True,
                       record_every=1 if QUICK else RECORD_EVERY)

In [ ]:
tag = run_tag(P, grid)
ckpt = os.path.join(RUN_DIR, f"checkpoint_{tag}.pkl")
if RESUME and os.path.exists(ckpt):
    sim.resume(ckpt)
    print(f"resumed from {ckpt} at pass {sim.passes_done}")
if not sim.done:
    t0 = time.time()
    sim.run(max_seconds=MAX_SECONDS)
    sim.checkpoint(ckpt)
    print(f"{time.time() - t0:.0f} s for this call")
print("finished" if sim.done else "time budget reached: run this cell again to continue")

## 6. Evolution along the cell

In [ ]:
h = sim.history
npass = np.array(h["pass"])
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2), layout="constrained")
ax[0].plot(npass, np.array(h["energy"]) / h["energy"][0], "o-"); ax[0].set_title("transmission")
ax[1].plot(npass, h["B"], "o-"); ax[1].set_title("B-integral, on-axis peak (rad)")
ax[2].plot(npass, np.array(h["w_peak_x"]) * 1e6, "o-", label="x")
ax[2].plot(npass, np.array(h["w_peak_y"]) * 1e6, "s-", ms=3, label="y")
ax[2].axhline(w0_ref * 1e6, color="k", ls=":", label="injected waist")
ax[2].set_title("beam radius at the cell centre (um)"); ax[2].legend(fontsize=8)
ax[3].plot(npass, h["eta00_total"], "o-", label="HG00 fraction")
ax[3].plot(npass, h["V_near"], "s-", label="homogeneity (2D, near field)")
ax[3].plot(npass, h["M2x_mean"], "^-", label="M2 x"); ax[3].legend(fontsize=8)
for a in ax:
    a.set_xlabel("pass")

## 7. Output spectrum
Three curves: the simulation, the authors' SPM model (Eq. 1, φ = 55 rad, applied to the 1.3 ps Gaussian), and optionally the measurement digitised from Fig. 2 into `data/kaumanns2018_spectrum.csv` (nm, linear intensity).

In [ ]:
XLIM_NM = (990, 1070)
lam, S = dg.total_spectrum(grid, sim.Aw)
meas = load_digitized("data/kaumanns2018_spectrum.csv")

# the paper's SPM model on the same grid: A(t) exp(i phi |A(t)|^2), Gaussian input
a = np.exp(-2 * np.log(2) * (grid.t / P["fwhm"]) ** 2)
E_spm = grid.hfft.ft_t(a * np.exp(1j * P["phi_fit"] * a ** 2))
idx = np.nonzero(grid.valid)[0]; idx = idx[np.argsort(grid.lam[idx])]
lam_spm = grid.lam[idx]
S_spm = np.abs(E_spm[idx]) ** 2 / lam_spm ** 2

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for a_, log in zip(ax, (False, True)):
    a_.plot(lam * 1e9, S / S.max(), label="simulation")
    a_.plot(lam_spm * 1e9, S_spm / S_spm.max(), "--", label=f"SPM model, phi = {P['phi_fit']:.0f} rad (paper)")
    if meas is not None:
        a_.plot(*meas, label="measured (digitised)")
    a_.set_xlim(*XLIM_NM); a_.set_xlabel("wavelength (nm)")
    if log:
        a_.set_yscale("log"); a_.set_ylim(1e-3, 2)
ax[0].legend(fontsize=8)

## 8. Compression

In [ ]:
Epix = beam_pixels(sim.Aw)
t_tl, P_tl = fourier_limit(grid, Epix)
t_c, P_c = integrated_power(grid, Epix, gdd_phase(grid, P["compressor_gdd"]))
best = dg.compress(grid, sim.Aw, gdd_bounds_fs2=(-20000, 0))
t_b, P_b = integrated_power(grid, Epix, gdd_phase(grid, best["gdd_fs2"]))

fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")
ax.plot(t_tl, P_tl / P_tl.max(), "k:", label=f"Fourier limit {fwhm(t_tl, P_tl):.1f} fs")
ax.plot(t_c, P_c / P_tl.max(), label=f"{P['compressor_gdd']:.0f} fs$^2$ (paper): {fwhm(t_c, P_c):.1f} fs, "
        f"{P_c.max()/P_tl.max()*100:.0f} %")
ax.plot(t_b, P_b / P_tl.max(), "--", label=f"best GDD {best['gdd_fs2']:.0f} fs$^2$: {fwhm(t_b, P_b):.1f} fs, "
        f"{P_b.max()/P_tl.max()*100:.0f} %")
ax.set_xlim(-300, 300); ax.set_xlabel("time (fs)"); ax.set_ylabel("power / Fourier limit"); ax.legend(fontsize=8)

## 9. Output beam: spatio-spectral homogeneity

In [ ]:
# near field (collimated output beam), along x: what the papers' imaging spectrometers measure
theta, lam_t, S_th = dg.spectrogram_theta_lambda(grid, sim.Aw, per_lambda=False)
V_x, V_mean = homogeneity_1d(S_th)
prof = S_th.sum(axis=1)
inside = prof > np.exp(-2) * prof.max()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
ax[0].pcolormesh(lam_t * 1e9, theta * 1e3, S_th / S_th.max(), shading="auto", cmap="inferno")
ax[0].set_xlim(*XLIM_NM); ax[0].set_ylim(theta[inside].min() * 2e3, theta[inside].max() * 2e3)
ax[0].set_xlabel("wavelength (nm)"); ax[0].set_ylabel("angle (mrad)"); ax[0].set_title("output S(theta_x, lambda)")
ax[1].plot(theta * 1e3, prof / prof.max(), "k", label="profile")
ax[1].plot(theta[inside] * 1e3, V_x[inside], "g", label="homogeneity V")
ax[1].set_xlim(ax[0].get_ylim()); ax[1].set_ylim(0, 1.05); ax[1].legend(fontsize=8)
ax[1].set_xlabel("angle (mrad)"); ax[1].set_title(f"weighted homogeneity (x): {V_mean*100:.1f} %")

## 10. Summary

In [ ]:
Tcell = sim.history["energy"][-1] / sim.history["energy"][0]
table([
    ("waist / mirror spot (um, 1/e^2 radius)", f"{lin['w0']*1e6:.0f} / {lin['w_mirror']*1e6:.0f}", f"{REF['w0_um']} / {REF['wm_um']}"),
    ("transmission of the cell",               f"{Tcell*100:.1f} %", f"{REF['T']*100:.1f} %"),
    ("Fourier limit (fs)",                     f"{fwhm(t_tl, P_tl):.1f}", f"{REF['tl_fs']}"),
    (f"compressed, {P['compressor_gdd']:.0f} fs^2 (fs)", f"{fwhm(t_c, P_c):.1f}", f"{REF['comp_fs']} (model 41.6)"),
    ("peak power vs Fourier limit",            f"{P_c.max()/P_tl.max()*100:.0f} %", f"{REF['peak']*100:.0f} % (model 67 %)"),
    ("best GDD (fs^2)",                        f"{best['gdd_fs2']:.0f}", f"{P['compressor_gdd']:.0f}"),
    ("M2 x / y (spectrally weighted)",         f"{sim.history['M2x_mean'][-1]:.2f} / {sim.history['M2y_mean'][-1]:.2f}", REF["M2"]),
    ("homogeneity, weighted (x)",              f"{V_mean*100:.1f} %", REF["V"]),
    ("B-integral, on-axis peak (rad)",         f"{sim.solver.B:.1f}", "55 (fit), 59 +- 6 (est.)"),
])
sim.save(os.path.join(RUN_DIR, "run.npz"), include_field=False)

## Checks before trusting the comparison
* **Convergence:** this grid is the coarsest of the three. Compare two presets and `PHI_MAX = 0.01` before drawing conclusions on M² or homogeneity.
* **B-integral:** the code reports the on-axis peak value, whereas the paper's 55 rad comes from fitting a plane-wave SPM model to the spatially integrated spectrum, and its estimate of 59 ± 6 rad is averaged over the beam. They are not the same quantity; compare the spectra instead.
* **n₂:** ±10 % in the paper. A ±10 % scan of `n2_1bar` brackets the spectral width.
* **Injection:** the experiment mode-matches to the linear eigenmode at P/Pc ≈ 0.5, so the beam size oscillates from pass to pass (plot 6). `injection="nonlinear"` shows the Kerr-matched case for comparison.